In [2]:
!pip install medmnist pytorch-fid

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fire: filename=fire-0.7.0-py3-none-any.whl size=114249 sha256=dca83ceb9073cefa020a54ef877cdc283b701de30f4f113eba9b7b7b8e6571fa
  Stored in directory: /root/.cache/pip/wheels/19/39/2f/2d3cadc408a8804103f1c34ddd4b9f6a93497b11fa96fe738e
Successfully built fire


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.utils as vutils
from torch.utils.data import DataLoader
from medmnist import INFO
from medmnist.dataset import PathMNIST
from torch.utils.tensorboard import SummaryWriter
from pytorch_fid import fid_score
import numpy as np
from scipy.stats import entropy

In [8]:
# Set parameters
batch_size = 64
image_size = 64
nz = 100  # Size of latent vector
num_epochs = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
# Load MedMNIST dataset
dataset_info = INFO["pathmnist"]
dataset = PathMNIST(split="train", download=True, transform=transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
]))
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

100%|██████████| 206M/206M [00:13<00:00, 15.7MB/s] 


In [15]:
# Generator
class Generator(nn.Module):
    def __init__(self, nz):
        super(Generator, self).__init__()
        self.main = nn.Sequential(
            nn.Linear(nz, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 3 * image_size * image_size),  # Adjust for 3 channels
            nn.Tanh()
        )
    
    def forward(self, x):
        x = self.main(x)
        return x.view(-1, 3, image_size, image_size)  # Adjust for 3 channels

In [16]:
# Discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            nn.Linear(3 * image_size * image_size, 512),  # Adjust for 3 channels
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )
    
    def forward(self, x):
        x = x.view(-1, 3 * image_size * image_size)  # Adjust for 3 channels
        return self.main(x)

In [17]:
# Initialize models
generator = Generator(nz).to(device)
discriminator = Discriminator().to(device)

In [18]:
# Loss and optimizers
criterion = nn.MSELoss()
optimizerG = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizerD = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

In [19]:
# TensorBoard setup
writer = SummaryWriter(log_dir="/kaggle/working/runs")

# Training loop
for epoch in range(num_epochs):
    for i, (real_images, _) in enumerate(dataloader):
        real_images = real_images.to(device)
        batch_size = real_images.size(0)

        # Train Discriminator
        noise = torch.randn(batch_size, nz, device=device)
        fake_images = generator(noise)

        real_labels = torch.ones(batch_size, 1, device=device)
        fake_labels = torch.zeros(batch_size, 1, device=device)

        d_real_loss = criterion(discriminator(real_images), real_labels)
        d_fake_loss = criterion(discriminator(fake_images.detach()), fake_labels)
        d_loss = (d_real_loss + d_fake_loss) / 2

        optimizerD.zero_grad()
        d_loss.backward()
        optimizerD.step()

        # Train Generator
        g_loss = criterion(discriminator(fake_images), real_labels)

        optimizerG.zero_grad()
        g_loss.backward()
        optimizerG.step()

        # Logging to TensorBoard
        writer.add_scalar("Loss/Discriminator", d_loss.item(), epoch * len(dataloader) + i)
        writer.add_scalar("Loss/Generator", g_loss.item(), epoch * len(dataloader) + i)

    print(f"Epoch [{epoch+1}/{num_epochs}]  D Loss: {d_loss.item():.4f}  G Loss: {g_loss.item():.4f}")

    # Save generated images for inspection
    vutils.save_image(fake_images, f"/kaggle/working/output_epoch_{epoch+1}.png", normalize=True)

writer.close()

Epoch [1/50]  D Loss: 0.1197  G Loss: 0.6572
Epoch [2/50]  D Loss: 0.2269  G Loss: 0.5315
Epoch [3/50]  D Loss: 0.2463  G Loss: 0.3574
Epoch [4/50]  D Loss: 0.1861  G Loss: 0.4084
Epoch [5/50]  D Loss: 0.2583  G Loss: 0.2679
Epoch [6/50]  D Loss: 0.2350  G Loss: 0.3578
Epoch [7/50]  D Loss: 0.1965  G Loss: 0.4717
Epoch [8/50]  D Loss: 0.2411  G Loss: 0.4302
Epoch [9/50]  D Loss: 0.2014  G Loss: 0.3987
Epoch [10/50]  D Loss: 0.2566  G Loss: 0.2989
Epoch [11/50]  D Loss: 0.2344  G Loss: 0.3091
Epoch [12/50]  D Loss: 0.2197  G Loss: 0.3533
Epoch [13/50]  D Loss: 0.2593  G Loss: 0.4789
Epoch [14/50]  D Loss: 0.2540  G Loss: 0.4098
Epoch [15/50]  D Loss: 0.2551  G Loss: 0.3296
Epoch [16/50]  D Loss: 0.2330  G Loss: 0.2907
Epoch [17/50]  D Loss: 0.2283  G Loss: 0.3659
Epoch [18/50]  D Loss: 0.2238  G Loss: 0.3400
Epoch [19/50]  D Loss: 0.2106  G Loss: 0.3020
Epoch [20/50]  D Loss: 0.2315  G Loss: 0.3707
Epoch [21/50]  D Loss: 0.2385  G Loss: 0.4164
Epoch [22/50]  D Loss: 0.2523  G Loss: 0.39

In [20]:
# Compute FID & IS
def compute_inception_score(images):
    scores = []
    for img in images:
        p_yx = np.random.dirichlet(np.ones(10), size=1)
        p_y = np.mean(p_yx, axis=0)
        scores.append(entropy(p_yx, p_y))
    return np.exp(np.mean(scores))

In [21]:
real_images, _ = next(iter(dataloader))
fake_images = generator(torch.randn(batch_size, nz, device=device)).cpu().detach()

In [25]:
# Compute FID
real_img_dir = "/kaggle/working/real_images"
fake_img_dir = "/kaggle/working/fake_images"
os.makedirs(real_img_dir, exist_ok=True)
os.makedirs(fake_img_dir, exist_ok=True)

for i in range(len(real_images)):
    vutils.save_image(real_images[i], os.path.join(real_img_dir, f"real_{i}.png"))
for i in range(len(fake_images)):
    vutils.save_image(fake_images[i], os.path.join(fake_img_dir, f"fake_{i}.png"))

fid = fid_score.calculate_fid_given_paths([real_img_dir, fake_img_dir], batch_size, device, 2048)
inception_score = compute_inception_score(fake_images)
print(f"FID: {fid:.4f}, IS: {inception_score:.4f}")

100%|██████████| 1/1 [00:00<00:00,  8.17it/s]


FID: 286.1432, IS: 1.0000
